# 植物图像检索索引系统 (CLIP ViT-B/32)

**作者**: Zihan Yin

**项目概述**: 本项目使用CLIP模型（ViT-B/32版本）为植物图像建立检索索引，支持基于图像相似度的植物种类检索。CLIP是一种多模态模型，能够将图像和文本映射到同一嵌入空间，非常适合图像检索任务。

## 步骤 1 — 安装依赖包

**目的**: 安装项目所需的Python库。

**说明**: 
- `torch` 和 `torchvision`: PyTorch深度学习框架，用于模型推理
- `transformers`: Hugging Face库，用于加载CLIP预训练模型
- `pillow`: 图像处理库
- `numpy`: 数值计算库
- `tqdm`: 进度条显示库
- `safetensors`: 安全的模型权重存储格式

**注意**: 使用`-q`参数静默安装，减少输出干扰。

In [ ]:
# # 静默安装PyTorch和torchvision（使用CUDA 12.1版本）
# %pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121

# # 静默安装其他依赖：transformers（CLIP模型）、pillow（图像处理）、numpy（数值计算）、tqdm（进度条）、safetensors（模型权重格式）
# %pip install -q transformers pillow numpy tqdm safetensors

## 步骤 2 — 导入库和配置参数

**目的**: 导入必要的Python库，设置项目路径、模型参数、随机种子等配置项。

**关键配置**:
| 参数 | 值 | 说明 |
|------|-----|------|
| MODEL_ID | openai/clip-vit-base-patch32 | CLIP模型版本 |
| NUM_AUGS | 20 | 每个类别的增强样本数 |
| BATCH_SIZE | 32 | 编码批次大小 |
| SEED | 42 | 随机种子（确保可复现） |

**设备检测**: 自动检测并使用GPU（CUDA），若无则回退到CPU。

In [ ]:
# ========== 导入库 ==========
from pathlib import Path                          # 路径处理
import re, json, time, random, math               # 正则、JSON、时间、随机数、数学
import numpy as np                                # 数值计算
from PIL import Image, ImageOps                   # 图像处理
from tqdm import tqdm                             # 进度条显示

import torch                                      # PyTorch框架
from torch import nn                              # 神经网络模块
from torchvision import transforms                # 图像变换/增强
from transformers import CLIPModel, CLIPProcessor # CLIP模型和处理器

# ========== 路径与参数配置 ==========
NOTEBOOK_DIR = Path(__file__).parent if "__file__" in globals() else Path().resolve()  # 笔记本所在目录
DATA_DIR     = (NOTEBOOK_DIR / "../../01_data_wrangling/01_raw_data/05_thumbnail_image").resolve()  # 数据目录
OUT_DIR      = Path("index")                       # 输出目录（嵌入向量&元数据）
MODEL_ID     = "openai/clip-vit-base-patch32"      # CLIP模型标识
MODEL_DIR    = Path("clip-vit-b32")                # 本地模型缓存目录
NUM_AUGS     = 20                                  # 每类别增强样本数（不含原图）
BATCH_SIZE   = 32                                  # 编码批次大小
SEED         = 42                                  # 随机种子（确保可复现）

OUT_DIR.mkdir(parents=True, exist_ok=True)         # 创建输出目录
MODEL_DIR.mkdir(parents=True, exist_ok=True)       # 创建模型缓存目录

FILE_RE = re.compile(r"plant_species_thumbnail_image_(\d+)\.jpg", re.IGNORECASE)  # 文件名正则（提取植物ID）

def set_seed(seed=42):
    """设置随机种子，确保实验可复现"""
    random.seed(seed)           # Python随机数
    np.random.seed(seed)        # NumPy随机数
    torch.manual_seed(seed)     # PyTorch随机数
    torch.cuda.manual_seed_all(seed)  # CUDA随机数（所有GPU）

set_seed(SEED)                                          # 设置随机种子
device = "cuda" if torch.cuda.is_available() else "cpu" # 检测设备（优先GPU）
device

'cuda'

### 快速查看数据集

**目的**: 验证数据目录是否存在，检查图像文件数量和文件名格式是否正确。

**检查项**:
1. 数据目录是否存在
2. JPG文件总数统计
3. 前5个文件名预览（验证命名规范）

**预期文件名格式**: `plant_species_thumbnail_image_数字.jpg`

In [ ]:
# 检查数据目录是否存在
print("目录是否存在？", DATA_DIR.exists())

# 统计JPG图像文件数量
print("找到的JPG文件数量:", len(list(DATA_DIR.glob("*.jpg"))))

# 显示前5个文件名（用于验证命名格式）
print("前5个文件:", [p.name for p in DATA_DIR.glob("*.jpg")][:5])

Directory exists? True
Number of JPGs found: 1319
First 5 files: ['plant_species_thumbnail_image_1.jpg', 'plant_species_thumbnail_image_10.jpg', 'plant_species_thumbnail_image_100.jpg', 'plant_species_thumbnail_image_1000.jpg', 'plant_species_thumbnail_image_1001.jpg']


## 步骤 3 — 工具函数（EXIF修正、轻度增强、文件扫描）

**目的**: 定义图像处理工具函数，确保输入数据质量一致。

**三个核心函数**:

1. **`exif_correct()`**: 
   - 修正照片EXIF方向信息（手机拍摄常有问题）
   - 将RGBA转换为RGB（白色背景）

2. **`light_aug_pipeline()`**: 
   - 轻度数据增强，避免原型漂移
   - 包括：翻转、颜色抖动、仿射变换、轻微模糊

3. **`parse_plant_images()`**: 
   - 扫描目录，提取植物ID和文件路径
   - 使用正则表达式匹配文件名

In [ ]:
def exif_correct(img: Image.Image) -> Image.Image:
    """
    修正图像的EXIF方向信息，并将RGBA转换为RGB格式
    
    参数:
        img: PIL Image对象
    
    返回:
        修正后的RGB图像
    """
    # 根据EXIF信息自动旋转图像到正确方向
    img = ImageOps.exif_transpose(img)
    
    # 如果图像是RGBA格式（带透明通道），转换为RGB（白色背景）
    if img.mode == "RGBA":
        bg = Image.new("RGB", img.size, (255, 255, 255))  # 创建白色背景
        bg.paste(img, mask=img.split()[-1])  # 使用alpha通道作为掩码粘贴
        return bg
    
    # 如果图像不是RGB格式，转换为RGB
    if img.mode != "RGB":
        img = img.convert("RGB")
    
    return img


def light_aug_pipeline():
    """
    轻度数据增强管道，用于避免原型漂移
    
    增强策略包括:
    - 随机水平翻转（50%概率）
    - 轻微的亮度/对比度/饱和度/色调抖动
    - 小角度旋转/平移 + 适度的尺度抖动
    - 低概率的轻度模糊
    """
    return transforms.Compose([
        transforms.RandomHorizontalFlip(p=0.5),  # 随机水平翻转
        transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15, hue=0.02),  # 颜色抖动
        transforms.RandomAffine(degrees=15, translate=(0.10, 0.10), scale=(0.9, 1.1)),  # 随机仿射变换
        transforms.RandomApply([transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0))], p=0.2),  # 随机高斯模糊
    ])


def parse_plant_images(data_dir: Path):
    """
    解析植物图像文件，提取植物ID和文件路径
    
    参数:
        data_dir: 数据目录路径
    
    返回:
        包含(植物ID, 文件路径)元组的列表
    """
    pairs = []
    # 遍历目录中的所有文件
    for p in sorted(data_dir.glob("*")):
        # 使用正则表达式匹配文件名
        m = FILE_RE.match(p.name)
        if m:
            plant_id = int(m.group(1))  # 提取植物ID
            pairs.append((plant_id, p))  # 添加到列表
    return pairs

# 解析植物图像文件
pairs = parse_plant_images(DATA_DIR)
print(f"找到 {len(pairs)} 个类别（每个类别一张缩略图）")

# 断言检查：确保找到了图像文件
assert len(pairs) > 0, "未找到符合命名模式的图像；请检查目录和文件名。"

Found 1319 classes (one thumbnail each)


## 步骤 4 — 加载CLIP模型和处理器（本地缓存）

**目的**: 从Hugging Face加载预训练的CLIP模型和图像处理器。

**关键操作**:
1. 加载 `CLIPModel` - 用于提取图像特征
2. 加载 `CLIPProcessor` - 用于图像预处理（resize、裁剪、归一化）
3. 设置模型为评估模式 (`eval()`)
4. 可选：缓存到本地目录，便于离线部署

**模型信息**: CLIP ViT-B/32 输出嵌入维度为 512

In [ ]:
print("正在加载CLIP模型和处理器…")

# 从Hugging Face加载预训练的CLIP模型
model = CLIPModel.from_pretrained(MODEL_ID)

# 加载CLIP图像处理器（用于图像预处理）
processor = CLIPProcessor.from_pretrained(MODEL_ID)

# 将模型设置为评估模式并移动到指定设备（GPU/CPU）
model.eval().to(device)

# 可选：将处理器缓存到本地，便于离线使用或部署
# model.save_pretrained(MODEL_DIR)  # 如需缓存模型可取消注释
processor.save_pretrained(MODEL_DIR)  # 保存处理器配置

# 获取嵌入向量的维度（CLIP ViT-B/32的输出维度为512）
embedding_dim = model.config.projection_dim
embedding_dim

Loading CLIP model and processor…


512

## 步骤 5 — 批量图像编码（GPU上使用混合精度）

**目的**: 定义批量编码函数，使用CLIP模型将图像转换为归一化的嵌入向量。

**技术要点**:
- `@torch.no_grad()`: 禁用梯度计算，节省内存
- 批量处理：避免内存溢出
- 混合精度 (`float16`): GPU加速，减少显存占用
- L2归一化：使向量长度为1，便于计算余弦相似度

**输入**: PIL图像列表  
**输出**: 归一化嵌入向量矩阵 (N, D)

In [ ]:
@torch.no_grad()  # 禁用梯度计算，节省内存并加速推理
def encode_images(pil_list, batch_size=32, amp_dtype=torch.float16):
    """
    使用CLIPProcessor进行标准预处理（调整大小/中心裁剪/归一化）
    返回L2归一化的图像嵌入向量
    
    参数:
        pil_list: PIL图像列表
        batch_size: 批次大小
        amp_dtype: 混合精度数据类型（默认float16）
    
    返回:
        归一化的嵌入向量矩阵 (N, D)，N为图像数量，D为嵌入维度
    """
    embs = []  # 存储每批次的嵌入向量
    
    # 按批次处理图像
    for i in range(0, len(pil_list), batch_size):
        batch = pil_list[i:i+batch_size]  # 获取当前批次
        
        # 使用CLIP处理器预处理图像
        inputs = processor(images=batch, return_tensors="pt", do_center_crop=True, padding=True)
        
        # 将输入数据移动到指定设备
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        # 使用混合精度加速（仅在GPU上启用）
        with torch.autocast(device_type=("cuda" if device=="cuda" else "cpu"),
                            dtype=amp_dtype, enabled=(device=="cuda")):
            feats = model.get_image_features(**inputs)  # 获取图像特征 (B, D)
        
        # L2归一化特征向量（使向量长度为1，便于计算余弦相似度）
        feats = nn.functional.normalize(feats, p=2, dim=-1)
        
        # 将结果移回CPU并添加到列表
        embs.append(feats.cpu())
    
    # 合并所有批次的嵌入向量
    return torch.cat(embs, dim=0)  # (N, D)

## 步骤 6 — 构建平均原型并缓存（主循环）

**目的**: 对每个植物类别的图像进行增强和编码，计算平均原型向量作为该类别的代表嵌入。

**处理流程**:
1. 打开图像并修正EXIF
2. 生成增强样本（原始图 + 20个增强版本）
3. 批量编码所有样本
4. 计算平均原型向量
5. L2归一化
6. 缓存到列表

**优势**: 使用多个增强样本的平均值作为原型，提高检索鲁棒性，减少单张图像的噪声影响。

In [ ]:
# 初始化数据增强管道
aug = light_aug_pipeline()

# 用于存储原型向量和植物ID的列表（跳过损坏的图像）
proto_list = []
plant_id_list = []

# 创建进度条，显示编码进度
pbar = tqdm(pairs, desc="正在编码类别（含数据增强）")

# 遍历每个植物类别
for plant_id, img_path in pbar:
    # 打开并修正图像
    try:
        img = Image.open(img_path)
    except Exception as e:
        print(f"[警告] 跳过损坏的图像 {img_path}: {e}")
        continue
    
    # 修正EXIF方向和颜色模式
    img = exif_correct(img)

    # 构建增强样本列表（包含原始图像 + 增强图像）
    pil_list = [img] + [aug(img) for _ in range(NUM_AUGS)]

    # 对增强后的图像进行批量编码
    feats = encode_images(pil_list, batch_size=BATCH_SIZE)  # (N, D)
    
    # 计算平均原型向量（所有增强样本的平均值）
    mean_proto = feats.mean(dim=0, keepdim=True)
    
    # L2归一化并转换为NumPy数组
    mean_proto = nn.functional.normalize(mean_proto, p=2, dim=-1).cpu().numpy()[0]  # (D,)

    # 缓存原型向量和植物ID
    proto_list.append(mean_proto.astype(np.float32))
    plant_id_list.append(plant_id)

# 组装嵌入矩阵和植物ID数组
embeddings = np.vstack(proto_list).astype(np.float32)  # (C, D)，C为类别数
plant_ids = np.array(plant_id_list, dtype=np.int64)

# 输出结果信息
print("嵌入矩阵形状:", embeddings.shape)
print("示例L2范数（应≈1）:", np.linalg.norm(embeddings[0]))

Encoding classes (with augmentation): 100%|██████████| 1319/1319 [11:35<00:00,  1.90it/s]

Embeddings shape: (1319, 512)
Example L2 norm (≈1): 0.99983567


## 步骤 7 — 保存索引（fp16格式）和元数据

**目的**: 将生成的嵌入向量和元数据保存到磁盘，便于后续检索使用。

**保存内容**:

1. **`embeddings_fp16.npz`** (压缩NPZ文件):
   - `embeddings`: fp16格式的嵌入矩阵 (类别数 × 512)
   - `plant_ids`: 对应的植物ID数组

2. **`meta.json`** (元数据文件):
   - 模型信息、嵌入维度、类别数量
   - 构建时间、设备信息、随机种子
   - 预处理配置、相似度计算方法

**为什么用fp16**: 节省50%存储空间，查询时可上转换为fp32，精度损失可忽略。

In [ ]:
# ========== 保存索引（fp16格式）和元数据 ==========
emb_fp16 = embeddings.astype(np.float16)                # 转换为fp16（节省50%存储空间）

np.savez_compressed(OUT_DIR / "embeddings_fp16.npz",    # 保存压缩NPZ文件
                    embeddings=emb_fp16,                # 嵌入向量（fp16）
                    plant_ids=plant_ids)                # 植物ID数组

meta = {
    "model_id": MODEL_ID,                               # CLIP模型标识
    "model_local_dir": str(MODEL_DIR.as_posix()),       # 本地模型缓存路径
    "embedding_dim": int(embedding_dim),                # 嵌入向量维度
    "num_classes": int(embeddings.shape[0]),            # 植物类别数量
    "num_augs_per_class": NUM_AUGS,                     # 每类别增强样本数
    "built_at": time.strftime("%Y-%m-%d %H:%M:%S"),     # 构建时间
    "device_used": device,                              # 使用设备（GPU/CPU）
    "seed": SEED,                                       # 随机种子
    "preprocess": {
        "center_crop": True,                            # 是否使用中心裁剪
        "note": "与CLIPProcessor默认设置对齐；在线查询必须匹配此预处理"
    },
    "similarity": "cosine (via dot on L2-normalized vectors)"  # 相似度计算方法
}

with open(OUT_DIR / "meta.json", "w", encoding="utf-8") as f:  # 保存元数据为JSON
    json.dump(meta, f, ensure_ascii=False, indent=2)

print("已保存:", (OUT_DIR / "embeddings_fp16.npz").as_posix())   # 输出保存路径
print("已保存:", (OUT_DIR / "meta.json").as_posix())
print("模型缓存位置:", MODEL_DIR.as_posix())

Saved: index/embeddings_fp16.npz
Saved: index/meta.json
Model cached at: clip-vit-b32


## 步骤 8 — 重新加载并验证

**目的**: 加载保存的索引文件，验证嵌入向量的形状和范数是否正确。

**验证项**:
1. 嵌入矩阵形状是否正确
2. 植物ID数组形状是否匹配
3. L2范数是否接近1（验证归一化是否成功）
4. 植物ID预览（确认数据顺序）

**成功标志**: 平均L2范数应接近 1.0，说明归一化正确。

In [ ]:
# 加载保存的索引文件
npz = np.load(OUT_DIR / "embeddings_fp16.npz")

# 读取嵌入向量并转换为float32
E = npz["embeddings"].astype(np.float32)  # (C, D)

# 读取植物ID数组
P = npz["plant_ids"]

# 验证加载结果
print("加载的嵌入向量形状:", E.shape, " 加载的植物ID形状:", P.shape)
print("平均L2范数:", np.linalg.norm(E, axis=1).mean())
print("前5个植物ID:", P[:5])

Loaded embeddings: (1319, 512)  Loaded plant_ids: (1319,)
Mean L2 norm: 1.0000056
First 5 plant_ids: [   1   10  100 1000 1001]
